# E5 GRPO from E3 step 300
Reward tests, 1/5-step smoke gates, full 60-step training, test inference and bonus evidence.
Only E5 runs; E0-E4 results are restored from the attached completed notebook output.


In [ ]:
import os, sys, json, shutil, subprocess, time, signal
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import torch
assert torch.cuda.is_available(), "Enable GPU T4 x2 in Kaggle Settings"
print("GPU:", torch.cuda.get_device_name(0), flush=True)
roots = sorted(p for p in Path("/kaggle/input").rglob("labrepo")
               if (p/"student/e5.py").is_file() and (p/"data/splits/train.jsonl").is_file())
assert roots, "Attach the completed E0-E4 notebook output as Input"
INPUT_ROOT = roots[0]
ROOT = Path("/kaggle/working/labrepo")
ROOT.mkdir(parents=True, exist_ok=True)
for name in ("scripts", "student", "student_tests", "configs", "data", "work", "report"):
    if (INPUT_ROOT/name).is_dir():
        shutil.copytree(INPUT_ROOT/name, ROOT/name, dirs_exist_ok=True)
for name in ("requirements-colab.txt", "pytest.ini", "REPORT.md"):
    if (INPUT_ROOT/name).is_file(): shutil.copy2(INPUT_ROOT/name, ROOT/name)
CHECKPOINT_ROOT = Path("/kaggle/working/day10_e5_observable_v1")
candidates = sorted(p for p in Path("/kaggle/input").rglob("step-0300")
                    if p.parent.name == "e3" and (p/"adapter/adapter_model.safetensors").is_file()
                    and (p/"projector.pt").is_file())
assert candidates, "Attach E3 step-0300 adapter and projector from the completed run"
target = CHECKPOINT_ROOT/"e3/step-0300"
shutil.copytree(candidates[0]/"adapter", target/"adapter", dirs_exist_ok=True)
shutil.copy2(candidates[0]/"projector.pt", target/"projector.pt")
if (candidates[0]/"run.json").is_file(): shutil.copy2(candidates[0]/"run.json", target/"run.json")
(CHECKPOINT_ROOT/"e3/latest.txt").write_text(str(target), encoding="utf-8")
os.environ["PYTHONPATH"] = str(ROOT)
os.chdir(ROOT)
meta = json.loads((ROOT/"data/splits/meta.json").read_text())
assert meta["dataset_version"] == "d10-21e4070301", meta
for split, expected in (("train",1422),("val",276),("test",311)):
    rows = [json.loads(x) for x in (ROOT/f"data/splits/{split}.jsonl").read_text().splitlines() if x.strip()]
    assert len(rows) == expected, (split,len(rows))
    assert all((ROOT/"data"/r["image_path"]).is_file() for r in rows), split
assert (ROOT/"work/e3_test.jsonl").is_file(), "Missing SFT baseline test predictions"
print("Dataset and E3 step 300 restored", flush=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(ROOT/"requirements-colab.txt")], check=True)
subprocess.run([sys.executable,"-m","pip","uninstall","-y","torchao"], capture_output=True)


In [ ]:
import base64, importlib.metadata
source = base64.b64decode('...').decode("utf-8")
compile(source, "student/e5.py", "exec")
(ROOT/"student/e5.py").write_text(source, encoding="utf-8")
source = base64.b64decode('IiIiRGV0ZXJtaW5pc3RpYyBiYXRjaGVkIGdlbmVyYXRpb24sIHNhZmUgcmV0cmllcywgYW5kIGdyYWRlciBpbnRlZ3JhdGlvbi4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCB0b3JjaApmcm9tIFBJTCBpbXBvcnQgSW1hZ2UKCmZyb20gc3R1ZGVudC5kYXRhIGltcG9ydCBQUk9NUFQsIGxvYWRfanNvbmwsIG1ha2VfbWVzc2FnZXMKZnJvbSBzdHVkZW50Lm1vZGVsIGltcG9ydCBsb2FkX2Jhc2VfbW9kZWwsIGxvYWRfdHJhaW5lZF9tb2RlbAoKCmRlZiBfbG9hZF9ydW4oZXhwLCBjaGVja3BvaW50KToKICAgIG1hcmtlciA9IFBhdGgoY2hlY2twb2ludCkgLyBleHAgLyAibGF0ZXN0LnR4dCIKICAgIGlmIG1hcmtlci5leGlzdHMoKToKICAgICAgICByZXR1cm4gUGF0aChtYXJrZXIucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpLnN0cmlwKCkpCiAgICBjaG9pY2VzID0gc29ydGVkKChQYXRoKGNoZWNrcG9pbnQpIC8gZXhwKS5nbG9iKCJzdGVwLSovYWRhcHRlciIpKQogICAgaWYgY2hvaWNlczoKICAgICAgICByZXR1cm4gY2hvaWNlc1stMV0ucGFyZW50CiAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIm5vIGNoZWNrcG9pbnQgZm91bmQgZm9yIHtleHB9IHVuZGVyIHtjaGVja3BvaW50fSIpCgoKZGVmIGdlbmVyYXRlX3NwbGl0KGV4cDogc3RyLCAqLCBzcGxpdD0idGVzdCIsIGRhdGFfcm9vdD0iZGF0YSIsIGNoZWNrcG9pbnRfcm9vdD0iL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS9kYXkxMCIsCiAgICAgICAgICAgICAgICAgICBiYXRjaF9zaXplPTQsIG91dHB1dD1Ob25lLCBtYXhfbmV3X3Rva2Vucz0yNTYsIHJlc3VtZT1GYWxzZSk6CiAgICBpZiBzcGxpdCBub3QgaW4geyJ2YWwiLCAidGVzdCJ9OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImluZmVyZW5jZSBzcGxpdCBtdXN0IGJlIHZhbCBvciB0ZXN0IikKICAgIGlmIGV4cCA9PSAiZTAiOgogICAgICAgIG1vZGVsID0gbG9hZF9iYXNlX21vZGVsKCkKICAgIGVsc2U6CiAgICAgICAgY2hlY2twb2ludCA9IF9sb2FkX3J1bihleHAsIGNoZWNrcG9pbnRfcm9vdCkKICAgICAgICBtb2RlbCA9IGxvYWRfdHJhaW5lZF9tb2RlbChleHAsIGNoZWNrcG9pbnQpCiAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b1Byb2Nlc3NvcgogICAgcHJvY2Vzc29yID0gQXV0b1Byb2Nlc3Nvci5mcm9tX3ByZXRyYWluZWQoIlF3ZW4vUXdlbjMtVkwtNEItSW5zdHJ1Y3QiKQogICAgcHJvY2Vzc29yLnRva2VuaXplci5wYWRkaW5nX3NpZGUgPSAibGVmdCIKICAgIHJvd3MgPSBsb2FkX2pzb25sKFBhdGgoZGF0YV9yb290KSAvICJzcGxpdHMiIC8gZiJ7c3BsaXR9Lmpzb25sIikKICAgIG91dHB1dCA9IFBhdGgob3V0cHV0IG9yIChQYXRoKCJ3b3JrIikgLyBmIntleHB9X3tzcGxpdH0uanNvbmwiKSkKICAgIG91dHB1dC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgbW9kZWwuZXZhbCgpCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGE6MCIpCiAgICBvdXRwdXRzID0gbG9hZF9qc29ubChvdXRwdXQpIGlmIHJlc3VtZSBhbmQgb3V0cHV0LmV4aXN0cygpIGVsc2UgW10KICAgIGlmIFt4WyJpbWFnZV9pZCJdIGZvciB4IGluIG91dHB1dHNdICE9IFt4WyJpbWFnZV9pZCJdIGZvciB4IGluIHJvd3NbOmxlbihvdXRwdXRzKV1dOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigicGFydGlhbCBwcmVkaWN0aW9ucyBhcmUgbm90IGFuIG9yZGVyZWQgcHJlZml4IG9mIHRoaXMgc3BsaXQiKQogICAgaWYgbm90IHJlc3VtZToKICAgICAgICBvdXRwdXQud3JpdGVfdGV4dCgiIiwgZW5jb2Rpbmc9InV0Zi04IikKICAgIGkgPSBsZW4ob3V0cHV0cykKICAgIGN1cnJlbnRfYnMgPSBiYXRjaF9zaXplCiAgICB3aGlsZSBpIDwgbGVuKHJvd3MpOgogICAgICAgIGdyb3VwID0gcm93c1tpOmkgKyBjdXJyZW50X2JzXQogICAgICAgIGltYWdlcyA9IFtdCiAgICAgICAgZm9yIHJvdyBpbiBncm91cDoKICAgICAgICAgICAgd2l0aCBJbWFnZS5vcGVuKFBhdGgoZGF0YV9yb290KSAvIHJvd1siaW1hZ2VfcGF0aCJdKSBhcyBpbToKICAgICAgICAgICAgICAgIGltYWdlcy5hcHBlbmQoaW0uY29udmVydCgiUkdCIikpCiAgICAgICAgbWVzc2FnZXMgPSBbbWFrZV9tZXNzYWdlcyhpbSkgZm9yIGltIGluIGltYWdlc10KICAgICAgICB0cnk6CiAgICAgICAgICAgIGlucHV0cyA9IHByb2Nlc3Nvci5hcHBseV9jaGF0X3RlbXBsYXRlKAogICAgICAgICAgICAgICAgbWVzc2FnZXMsIHRva2VuaXplPVRydWUsIGFkZF9nZW5lcmF0aW9uX3Byb21wdD1UcnVlLCByZXR1cm5fZGljdD1UcnVlLAogICAgICAgICAgICAgICAgcmV0dXJuX3RlbnNvcnM9InB0IiwgcGFkZGluZz1UcnVlLAogICAgICAgICAgICApCiAgICAgICAgICAgIGlucHV0cy5wb3AoInRva2VuX3R5cGVfaWRzIiwgTm9uZSkKICAgICAgICAgICAgaW5wdXRzID0ge2s6IHYudG8oZGV2aWNlKSBpZiB0b3JjaC5pc190ZW5zb3IodikgZWxzZSB2IGZvciBrLCB2IGluIGlucHV0cy5pdGVtcygpfQogICAgICAgICAgICB3aXRoIHRvcmNoLmluZmVyZW5jZV9tb2RlKCk6CiAgICAgICAgICAgICAgICBnZW5lcmF0ZWQgPSBtb2RlbC5nZW5lcmF0ZSgqKmlucHV0cywgbWF4X25ld190b2tlbnM9bWF4X25ld190b2tlbnMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBkb19zYW1wbGU9RmFsc2UsIG51bV9iZWFtcz0xKQogICAgICAgICAgICBwcm9tcHRfbGVuID0gaW5wdXRzWyJpbnB1dF9pZHMiXS5zaGFwZVsxXQogICAgICAgICAgICBkZWNvZGVkID0gW3Byb2Nlc3Nvci5kZWNvZGUoZ1twcm9tcHRfbGVuOl0sIHNraXBfc3BlY2lhbF90b2tlbnM9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNsZWFuX3VwX3Rva2VuaXphdGlvbl9zcGFjZXM9RmFsc2UpCiAgICAgICAgICAgICAgICAgICAgICAgZm9yIGcgaW4gZ2VuZXJhdGVkXQogICAgICAgICAgICBiYXRjaF9vdXRwdXRzID0gW3siaW1hZ2VfaWQiOiByb3dbImltYWdlX2lkIl0sICJyYXciOiByYXd9CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIHJvdywgcmF3IGluIHppcChncm91cCwgZGVjb2RlZCldCiAgICAgICAgICAgIG91dHB1dHMuZXh0ZW5kKGJhdGNoX291dHB1dHMpCiAgICAgICAgICAgIHdpdGggb3V0cHV0Lm9wZW4oImEiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgICAgICAgICAgZm9yIGl0ZW0gaW4gYmF0Y2hfb3V0cHV0czoKICAgICAgICAgICAgICAgICAgICBmLndyaXRlKGpzb24uZHVtcHMoaXRlbSwgZW5zdXJlX2FzY2lpPUZhbHNlKSArICJcbiIpCiAgICAgICAgICAgICAgICBmLmZsdXNoKCkKICAgICAgICAgICAgaSArPSBsZW4oZ3JvdXApCiAgICAgICAgICAgIHByaW50KGYiSU5GRVJFTkNFX1BST0dSRVNTIHtleHB9IHtzcGxpdH0ge2l9L3tsZW4ocm93cyl9IiwgZmx1c2g9VHJ1ZSkKICAgICAgICBleGNlcHQgdG9yY2guY3VkYS5PdXRPZk1lbW9yeUVycm9yOgogICAgICAgICAgICB0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKCkKICAgICAgICAgICAgaWYgY3VycmVudF9icyA9PSAxOgogICAgICAgICAgICAgICAgcmFpc2UKICAgICAgICAgICAgY3VycmVudF9icyA9IG1heCgxLCBjdXJyZW50X2JzIC8vIDIpCiAgICAgICAgICAgIHByaW50KGYiT09NOyByZXRyeSBpbmZlcmVuY2UgYXQgYmF0Y2hfc2l6ZT17Y3VycmVudF9ic30iKQogICAgaWYgbGVuKG91dHB1dHMpICE9IGxlbihyb3dzKSBvciBsZW4oe3hbImltYWdlX2lkIl0gZm9yIHggaW4gb3V0cHV0c30pICE9IGxlbihyb3dzKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoImluZmVyZW5jZSBkaWQgbm90IHByb2R1Y2UgZXhhY3RseSBvbmUgcmVzdWx0IHBlciBzcGxpdCBpbWFnZSIpCiAgICByZXR1cm4gb3V0cHV0CgoKZGVmIHNjb3JlX3ByZWRpY3Rpb25zKGV4cDogc3RyLCBwcmVkX3BhdGgsICosIHNwbGl0PSJ0ZXN0IiwgZGF0YV9yb290PSJkYXRhIiwgb3V0PU5vbmUsCiAgICAgICAgICAgICAgICAgICAgICB0cmFpbl9ydW49Tm9uZSwgY29vcmQ9Im5vcm0xMDAwIik6CiAgICBwcmVkX3BhdGggPSBQYXRoKHByZWRfcGF0aCkKICAgIGlmIG91dCBpcyBOb25lOgogICAgICAgIG91dCA9IFBhdGgoInJlcG9ydC9ldmlkZW5jZSIpIC8gZiJ7ZXhwfS5qc29uIiBpZiBzcGxpdCA9PSAidGVzdCIgZWxzZSBQYXRoKCJ3b3JrIikgLyBmIntleHB9X3tzcGxpdH1fc2NvcmUuanNvbiIKICAgIGFyZ3MgPSBbc3lzLmV4ZWN1dGFibGUsICJzY3JpcHRzL3Njb3JlLnB5IiwgIi0tcHJlZCIsIHN0cihwcmVkX3BhdGgpLAogICAgICAgICAgICAiLS1zcGxpdHMtZGlyIiwgc3RyKFBhdGgoZGF0YV9yb290KSAvICJzcGxpdHMiKSwgIi0tc3BsaXQiLCBzcGxpdCwKICAgICAgICAgICAgIi0tY29vcmQiLCBjb29yZCwgIi0tbmFtZSIsIGV4cC51cHBlcigpLCAiLS1vdXQiLCBzdHIob3V0KV0KICAgIGlmIHRyYWluX3J1bjoKICAgICAgICBydW4gPSBqc29uLmxvYWRzKFBhdGgodHJhaW5fcnVuKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgZm9yIGtleSBpbiAoInRyYWluYWJsZV9wYXJhbXMiLCAicGVha192cmFtX2diIiwgInRyYWluX21pbnV0ZXMiKToKICAgICAgICAgICAgdmFsID0gcnVuLmdldChrZXkpCiAgICAgICAgICAgIGlmIHZhbCBpcyBOb25lIG9yIGZsb2F0KHZhbCkgPD0gMDoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ0cmFpbmluZyBydW4gbWlzc2luZyBwb3NpdGl2ZSB7a2V5fSIpCiAgICAgICAgICAgIGFyZ3MuZXh0ZW5kKFsiLS1tZXRhIiwgZiJ7a2V5fT17dmFsfSJdKQogICAgc3VicHJvY2Vzcy5ydW4oYXJncywgY2hlY2s9VHJ1ZSkKICAgIHJldHVybiBQYXRoKG91dCkK').decode("utf-8")
compile(source, "student/inference.py", "exec")
(ROOT/"student/inference.py").write_text(source, encoding="utf-8")
source = base64.b64decode('IiIiUXdlbjMtVkwgbG9hZGluZyBhbmQgdGhlIGxhYidzIEUxL0UyL0UzIFFMb1JBIHNldHVwcy4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHJlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHRvcmNoCgpNT0RFTF9JRCA9ICJRd2VuL1F3ZW4zLVZMLTRCLUluc3RydWN0IgpMTE1fVEFSR0VUUyA9IHJlLmNvbXBpbGUoCiAgICByIl4oPzptb2RlbFwuKT9sYW5ndWFnZV9tb2RlbFwubGF5ZXJzXC5cZCtcLig/OnNlbGZfYXR0blwuKD86cXxrfHZ8bylfcHJvanwiCiAgICByIm1scFwuKD86Z2F0ZXx1cHxkb3duKV9wcm9qKSQiCikKVklUX1RBUkdFVFMgPSByZS5jb21waWxlKAogICAgciJeKD86bW9kZWxcLik/dmlzdWFsXC5ibG9ja3NcLlxkK1wuKD86YXR0blwuKD86cWt2fHByb2opfG1scFwuKD86bGluZWFyX2ZjMXxsaW5lYXJfZmMyKSkkIgopClBST0pFQ1RPUl9QQVJUUyA9ICgidmlzdWFsLm1lcmdlciIsICJ2aXN1YWwuZGVlcHN0YWNrX21lcmdlcl9saXN0IikKRlVMTF9QUkVDSVNJT05fU0tJUF9NT0RVTEVTID0gWyJ2aXN1YWwiLCAibW9kZWwudmlzdWFsIiwgImxtX2hlYWQiLCAibW9kZWwubG1faGVhZCJdCgoKZGVmIHNldF9zZWVkKHNlZWQ9NDIpOgogICAgaW1wb3J0IHJhbmRvbQogICAgaW1wb3J0IG51bXB5IGFzIG5wCiAgICByYW5kb20uc2VlZChzZWVkKQogICAgbnAucmFuZG9tLnNlZWQoc2VlZCkKICAgIHRvcmNoLm1hbnVhbF9zZWVkKHNlZWQpCiAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHRvcmNoLmN1ZGEubWFudWFsX3NlZWRfYWxsKHNlZWQpCgoKZGVmIGxvYWRfYmFzZV9tb2RlbChtb2RlbF9pZD1NT0RFTF9JRCk6CiAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQml0c0FuZEJ5dGVzQ29uZmlnLCBRd2VuM1ZMRm9yQ29uZGl0aW9uYWxHZW5lcmF0aW9uCiAgICBxdWFudCA9IEJpdHNBbmRCeXRlc0NvbmZpZygKICAgICAgICBsb2FkX2luXzRiaXQ9VHJ1ZSwgYm5iXzRiaXRfcXVhbnRfdHlwZT0ibmY0IiwKICAgICAgICBibmJfNGJpdF9jb21wdXRlX2R0eXBlPXRvcmNoLmZsb2F0MTYsIGJuYl80Yml0X3VzZV9kb3VibGVfcXVhbnQ9VHJ1ZSwKICAgICAgICAjIFRyYW5zZm9ybWVycyB1c2VzIHRoZXNlIHNraXAgbmFtZXMgZm9yIGJvdGggdGhlIDgtYml0IGFuZCA0LWJpdAogICAgICAgICMgYml0c2FuZGJ5dGVzIHBhdGhzLiBRd2VuMy1WTCB2ZXJzaW9ucyBkaWZmZXIgaW4gd2hldGhlciBtb2R1bGVzIGFyZQogICAgICAgICMgcm9vdGVkIGF0IHZpc3VhbC4qIG9yIG1vZGVsLnZpc3VhbC4qLCBzbyBrZWVwIGJvdGggc3BlbGxpbmdzLgogICAgICAgIGxsbV9pbnQ4X3NraXBfbW9kdWxlcz1GVUxMX1BSRUNJU0lPTl9TS0lQX01PRFVMRVMsCiAgICApCiAgICBtb2RlbCA9IFF3ZW4zVkxGb3JDb25kaXRpb25hbEdlbmVyYXRpb24uZnJvbV9wcmV0cmFpbmVkKAogICAgICAgIG1vZGVsX2lkLCBxdWFudGl6YXRpb25fY29uZmlnPXF1YW50LCBkdHlwZT10b3JjaC5mbG9hdDE2LAogICAgICAgIGRldmljZV9tYXA9eyIiOiAwfSwgYXR0bl9pbXBsZW1lbnRhdGlvbj0ic2RwYSIsCiAgICApCiAgICB2YWxpZGF0ZV9xdWFudGl6YXRpb25fbGF5b3V0KG1vZGVsKQogICAgcmV0dXJuIG1vZGVsCgoKZGVmIGluc3BlY3RfYXJjaGl0ZWN0dXJlKG1vZGVsKToKICAgICIiIkNvdW50IGxvZ2ljYWwgcGFyYW1ldGVycywgaW5jbHVkaW5nIHVucGFja2VkIHNoYXBlcyBvZiBORjQgbGluZWFycy4iIiIKICAgIG1vZHVsZXMgPSBkaWN0KG1vZGVsLm5hbWVkX21vZHVsZXMoKSkKICAgIHRvdGFscyA9IHsibGxtIjogMCwgInZpdCI6IDAsICJwcm9qZWN0b3IiOiAwLCAidG90YWwiOiAwfQogICAgZm9yIG5hbWUsIHAgaW4gbW9kZWwubmFtZWRfcGFyYW1ldGVycygpOgogICAgICAgIG1vZHVsZSA9IG1vZHVsZXMuZ2V0KG5hbWUucnNwbGl0KCIuIiwgMSlbMF0pCiAgICAgICAgaWYgKG5hbWUuZW5kc3dpdGgoIi53ZWlnaHQiKSBhbmQgbW9kdWxlIGlzIG5vdCBOb25lCiAgICAgICAgICAgICAgICBhbmQgbW9kdWxlLl9fY2xhc3NfXy5fX25hbWVfXyA9PSAiTGluZWFyNGJpdCIpOgogICAgICAgICAgICBuID0gaW50KG1vZHVsZS5pbl9mZWF0dXJlcykgKiBpbnQobW9kdWxlLm91dF9mZWF0dXJlcykKICAgICAgICBlbHNlOgogICAgICAgICAgICBuID0gcC5udW1lbCgpCiAgICAgICAgdG90YWxzWyJ0b3RhbCJdICs9IG4KICAgICAgICBpZiBhbnkocGFydCBpbiBuYW1lIGZvciBwYXJ0IGluIFBST0pFQ1RPUl9QQVJUUyk6CiAgICAgICAgICAgIHRvdGFsc1sicHJvamVjdG9yIl0gKz0gbgogICAgICAgIGVsaWYgbmFtZS5zdGFydHN3aXRoKCgidmlzdWFsLiIsICJtb2RlbC52aXN1YWwuIikpOgogICAgICAgICAgICB0b3RhbHNbInZpdCJdICs9IG4KICAgICAgICBlbHNlOgogICAgICAgICAgICB0b3RhbHNbImxsbSJdICs9IG4KICAgIHJldHVybiB0b3RhbHMKCgpkZWYgdmFsaWRhdGVfcXVhbnRpemF0aW9uX2xheW91dChtb2RlbCk6CiAgICAiIiJGYWlsIGJlZm9yZSB0cmFpbmluZyBpZiBhbnkgdmlzaW9uL3Byb2plY3RvciBsaW5lYXIgd2FzIHF1YW50aXplZC4iIiIKICAgIGJhZF92aXN1YWwgPSBbXQogICAgYmFkX2hlYWQgPSBbXQogICAgZm9yIG5hbWUsIG1vZHVsZSBpbiBtb2RlbC5uYW1lZF9tb2R1bGVzKCk6CiAgICAgICAgaWYgbW9kdWxlLl9fY2xhc3NfXy5fX25hbWVfXyAhPSAiTGluZWFyNGJpdCI6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgbmFtZS5zdGFydHN3aXRoKCgidmlzdWFsLiIsICJtb2RlbC52aXN1YWwuIikpOgogICAgICAgICAgICBiYWRfdmlzdWFsLmFwcGVuZChuYW1lKQogICAgICAgIGlmIG5hbWUgaW4geyJsbV9oZWFkIiwgIm1vZGVsLmxtX2hlYWQifToKICAgICAgICAgICAgYmFkX2hlYWQuYXBwZW5kKG5hbWUpCiAgICBpZiBiYWRfdmlzdWFsOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgInZpc2lvbiB0b3dlciB3YXMgcXVhbnRpemVkIGRlc3BpdGUgdGhlIHNraXAgbGlzdDogIgogICAgICAgICAgICArICIsICIuam9pbihiYWRfdmlzdWFsWzo1XSkpCiAgICBpZiBiYWRfaGVhZDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoImxtX2hlYWQgd2FzIHF1YW50aXplZDogIiArICIsICIuam9pbihiYWRfaGVhZCkpCgogICAgdG90YWxzID0gaW5zcGVjdF9hcmNoaXRlY3R1cmUobW9kZWwpCiAgICBpZiBub3QgNF8zMDBfMDAwXzAwMCA8PSB0b3RhbHNbInRvdGFsIl0gPD0gNF82MDBfMDAwXzAwMDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ1bmV4cGVjdGVkIGxvZ2ljYWwgdG90YWwgcGFyYW1ldGVyIGNvdW50OiB7dG90YWxzfSIpCiAgICBpZiBub3QgNDAwXzAwMF8wMDAgPD0gdG90YWxzWyJ2aXQiXSArIHRvdGFsc1sicHJvamVjdG9yIl0gPD0gNDMwXzAwMF8wMDA6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYidW5leHBlY3RlZCB2aXNpb24gcGFyYW1ldGVyIGNvdW50OiB7dG90YWxzfSIpCiAgICBpZiBub3QgMTAwXzAwMF8wMDAgPD0gdG90YWxzWyJwcm9qZWN0b3IiXSA8PSAxMTVfMDAwXzAwMDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJwcm9qZWN0b3IgaXMgbm90IGZ1bGwgcHJlY2lzaW9uL3VucGFja2VkOiB7dG90YWxzfSIpCiAgICBpZiBub3QgM185MDBfMDAwXzAwMCA8PSB0b3RhbHNbImxsbSJdIDw9IDRfMTAwXzAwMF8wMDA6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYidW5leHBlY3RlZCBsYW5ndWFnZS1tb2RlbCBwYXJhbWV0ZXIgY291bnQ6IHt0b3RhbHN9IikKICAgIHJldHVybiB0b3RhbHMKCgpkZWYgYXBwbHlfc2V0dXAobW9kZWwsIGV4cDogc3RyLCAqLCByYW5rPTE2KToKICAgICIiIkZyZWV6ZSBiYXNlIHdlaWdodHMsIGluc2VydCBvbmx5IHRoZSByZXF1ZXN0ZWQgTExNL1ZpVCBhZGFwdGVycy4iIiIKICAgIGZyb20gcGVmdCBpbXBvcnQgTG9yYUNvbmZpZywgZ2V0X3BlZnRfbW9kZWwsIHByZXBhcmVfbW9kZWxfZm9yX2tiaXRfdHJhaW5pbmcKCiAgICBpZiBleHAgbm90IGluIHsiZTEiLCAiZTIiLCAiZTMifToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5zdXBwb3J0ZWQgU0ZUIGV4cGVyaW1lbnQ6IHtleHB9IikKICAgIGFyY2hpdGVjdHVyZV9wYXJhbXMgPSBpbnNwZWN0X2FyY2hpdGVjdHVyZShtb2RlbCkKICAgICMgQ29uZmlndXJlIGNoZWNrcG9pbnRpbmcgZXhwbGljaXRseS4gUEVGVCdzIGNvbnZlbmllbmNlIGZsYWcgY3VycmVudGx5CiAgICAjIGVuYWJsZXMgdGhlIFRyYW5zZm9ybWVycyBkZWZhdWx0IChyZWVudHJhbnQpLCB3aGljaCBlbWl0cyB3YXJuaW5ncyBhbmQKICAgICMgY2FuIGZhaWwgd2hlbiBhIGZyb3plbiB2aXNpb24gYmxvY2sgcmVjZWl2ZXMgbm8gZ3JhZC1yZXF1aXJpbmcgaW5wdXQuCiAgICBtb2RlbCA9IHByZXBhcmVfbW9kZWxfZm9yX2tiaXRfdHJhaW5pbmcobW9kZWwsIHVzZV9ncmFkaWVudF9jaGVja3BvaW50aW5nPUZhbHNlKQogICAgbW9kZWwuZ3JhZGllbnRfY2hlY2twb2ludGluZ19lbmFibGUoZ3JhZGllbnRfY2hlY2twb2ludGluZ19rd2FyZ3M9eyJ1c2VfcmVlbnRyYW50IjogRmFsc2V9KQogICAgZm9yIHAgaW4gbW9kZWwucGFyYW1ldGVycygpOgogICAgICAgIHAucmVxdWlyZXNfZ3JhZF8oRmFsc2UpCiAgICBuYW1lcyA9IFtuYW1lIGZvciBuYW1lLCBfbW9kdWxlIGluIG1vZGVsLm5hbWVkX21vZHVsZXMoKQogICAgICAgICAgICAgaWYgTExNX1RBUkdFVFMuZnVsbG1hdGNoKG5hbWUpIG9yIChleHAgPT0gImUzIiBhbmQgVklUX1RBUkdFVFMuZnVsbG1hdGNoKG5hbWUpKV0KICAgIGV4cGVjdGVkID0gMzQ4IGlmIGV4cCA9PSAiZTMiIGVsc2UgMjUyCiAgICBpZiBsZW4obmFtZXMpICE9IGV4cGVjdGVkOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntleHB9OiBleHBlY3RlZCB7ZXhwZWN0ZWR9IExvUkEgbW9kdWxlcywgZm91bmQge2xlbihuYW1lcyl9IikKICAgIG1vZGVsLmVuYWJsZV9pbnB1dF9yZXF1aXJlX2dyYWRzKCkKICAgIG1vZGVsLmNvbmZpZy51c2VfY2FjaGUgPSBGYWxzZQogICAgY29uZmlnID0gTG9yYUNvbmZpZyhyPXJhbmssIGxvcmFfYWxwaGE9MiAqIHJhbmssIGxvcmFfZHJvcG91dD0wLjA1LAogICAgICAgICAgICAgICAgICAgICAgICB0YXJnZXRfbW9kdWxlcz1uYW1lcywgYmlhcz0ibm9uZSIsIHRhc2tfdHlwZT0iQ0FVU0FMX0xNIikKICAgIHBlZnRfbW9kZWwgPSBnZXRfcGVmdF9tb2RlbChtb2RlbCwgY29uZmlnKQogICAgaWYgZXhwIGluIHsiZTIiLCAiZTMifToKICAgICAgICBwcm9qZWN0b3JfbiA9IDAKICAgICAgICBmb3IgbmFtZSwgcCBpbiBwZWZ0X21vZGVsLm5hbWVkX3BhcmFtZXRlcnMoKToKICAgICAgICAgICAgaWYgYW55KHBhcnQgaW4gbmFtZSBmb3IgcGFydCBpbiBQUk9KRUNUT1JfUEFSVFMpOgogICAgICAgICAgICAgICAgcC5kYXRhID0gcC5kYXRhLnRvKHRvcmNoLmZsb2F0MzIpCiAgICAgICAgICAgICAgICBwLnJlcXVpcmVzX2dyYWRfKFRydWUpCiAgICAgICAgICAgICAgICBwcm9qZWN0b3JfbiArPSBwLm51bWVsKCkKICAgICAgICBpZiBwcm9qZWN0b3JfbiA9PSAwOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoInByb2plY3RvciBwYXJhbWV0ZXJzIHdlcmUgbm90IGZvdW5kIikKICAgICAgICBpZiBub3QgMTAwXzAwMF8wMDAgPD0gcHJvamVjdG9yX24gPD0gMTE1XzAwMF8wMDA6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgICAgIGYidW5leHBlY3RlZCBmdWxsIHByb2plY3RvciBzaXplIGZvciBRd2VuMy1WTC00Qjoge3Byb2plY3Rvcl9uOix9IHBhcmFtZXRlcnMiKQogICAgIyBHcmFkU2NhbGVyIHJlamVjdHMgRlAxNiBwYXJhbWV0ZXIgZ3JhZGllbnRzIGR1cmluZyB1bnNjYWxlXy4gS2VlcCBzbWFsbAogICAgIyB0cmFpbmFibGUgTG9SQSB3ZWlnaHRzIGluIEZQMzIgKGJhc2Ugd2VpZ2h0cyByZW1haW4gTkY0L0ZQMTYpOyB0aGlzIGFsc28KICAgICMgZ2l2ZXMgdGhlIGZ1bGwgRTIvRTMgcHJvamVjdG9yIGEgY29uc2lzdGVudCBGUDMyIG9wdGltaXplciBzdGF0ZS4KICAgIGZvciBwYXJhbWV0ZXIgaW4gcGVmdF9tb2RlbC5wYXJhbWV0ZXJzKCk6CiAgICAgICAgaWYgcGFyYW1ldGVyLnJlcXVpcmVzX2dyYWQgYW5kIHBhcmFtZXRlci5kdHlwZSA9PSB0b3JjaC5mbG9hdDE2OgogICAgICAgICAgICBwYXJhbWV0ZXIuZGF0YSA9IHBhcmFtZXRlci5kYXRhLmZsb2F0KCkKICAgIHJldHVybiBwZWZ0X21vZGVsLCB7Im1hdGNoZWRfbW9kdWxlcyI6IGxlbihuYW1lcyksICJ0cmFpbmFibGVfcGFyYW1zIjogc3VtKAogICAgICAgIHAubnVtZWwoKSBmb3IgcCBpbiBwZWZ0X21vZGVsLnBhcmFtZXRlcnMoKSBpZiBwLnJlcXVpcmVzX2dyYWQpLAogICAgICAgICJhcmNoaXRlY3R1cmVfcGFyYW1zIjogYXJjaGl0ZWN0dXJlX3BhcmFtc30KCgpkZWYgc2F2ZV9wcm9qZWN0b3IobW9kZWwsIHBhdGgpOgogICAgc3RhdGUgPSB7bjogcC5kZXRhY2goKS5jcHUoKSBmb3IgbiwgcCBpbiBtb2RlbC5uYW1lZF9wYXJhbWV0ZXJzKCkKICAgICAgICAgICAgIGlmIGFueShwYXJ0IGluIG4gZm9yIHBhcnQgaW4gUFJPSkVDVE9SX1BBUlRTKX0KICAgIGlmIG5vdCBzdGF0ZToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIm5vIHByb2plY3RvciBwYXJhbWV0ZXJzIHRvIHNhdmUiKQogICAgUGF0aChwYXRoKS5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgdG9yY2guc2F2ZShzdGF0ZSwgcGF0aCkKCgpkZWYgbG9hZF9wcm9qZWN0b3IobW9kZWwsIHBhdGgpOgogICAgc3RhdGUgPSB0b3JjaC5sb2FkKHBhdGgsIG1hcF9sb2NhdGlvbj0iY3B1Iiwgd2VpZ2h0c19vbmx5PVRydWUpCiAgICBwYXJhbXMgPSBkaWN0KG1vZGVsLm5hbWVkX3BhcmFtZXRlcnMoKSkKICAgIG1pc3NpbmcgPSBzZXQoc3RhdGUpIC0gc2V0KHBhcmFtcykKICAgIGlmIG1pc3Npbmc6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYicHJvamVjdG9yIGNoZWNrcG9pbnQgaGFzIHVua25vd24ga2V5czoge3NvcnRlZChtaXNzaW5nKVs6M119IikKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIGZvciBuYW1lLCB0ZW5zb3IgaW4gc3RhdGUuaXRlbXMoKToKICAgICAgICAgICAgcGFyYW1zW25hbWVdLmNvcHlfKHRlbnNvci50byhkZXZpY2U9cGFyYW1zW25hbWVdLmRldmljZSwgZHR5cGU9cGFyYW1zW25hbWVdLmR0eXBlKSkKCgpkZWYgbG9hZF90cmFpbmVkX21vZGVsKGV4cCwgY2hlY2twb2ludCk6CiAgICBmcm9tIHBlZnQgaW1wb3J0IFBlZnRNb2RlbAogICAgYmFzZSA9IGxvYWRfYmFzZV9tb2RlbCgpCiAgICBhZGFwdGVyX3BhdGggPSBQYXRoKGNoZWNrcG9pbnQpIC8gImFkYXB0ZXIiCiAgICBtb2RlbCA9IFBlZnRNb2RlbC5mcm9tX3ByZXRyYWluZWQoYmFzZSwgYWRhcHRlcl9wYXRoLCBpc190cmFpbmFibGU9RmFsc2UpCiAgICBwcm9qZWN0b3JfcGF0aCA9IFBhdGgoY2hlY2twb2ludCkgLyAicHJvamVjdG9yLnB0IgogICAgaWYgcHJvamVjdG9yX3BhdGguZXhpc3RzKCk6CiAgICAgICAgbG9hZF9wcm9qZWN0b3IobW9kZWwsIHByb2plY3Rvcl9wYXRoKQogICAgcmV0dXJuIG1vZGVsCg==').decode("utf-8")
compile(source, "student/model.py", "exec")
(ROOT/"student/model.py").write_text(source, encoding="utf-8")
versions = {p: importlib.metadata.version(p) for p in
            ("torch","transformers","peft","trl","bitsandbytes","accelerate")}
print("PACKAGES", json.dumps(versions), flush=True)
print("E5 source replaced: plain logs, per-step progress, incremental predictions", flush=True)


In [ ]:
LOG_ROOT = Path("/kaggle/working/e5_job_logs")
LOG_ROOT.mkdir(exist_ok=True)
def run_job(name, args, *, inactivity=900, total_timeout=7200):
    logfile = LOG_ROOT/f"{name}.log"
    # The child writes to a regular file, avoiding notebook stdout pipe backpressure.
    launcher = "import faulthandler,runpy; faulthandler.enable(); faulthandler.dump_traceback_later(600,repeat=True); runpy.run_module('student.e5',run_name='__main__')"
    command = [sys.executable,"-u","-c",launcher,*args]
    start = last_progress = last_heartbeat = time.monotonic()
    print("START", name, args, flush=True)
    with logfile.open("wb", buffering=0) as sink:
        child = subprocess.Popen(command, cwd=ROOT, env=os.environ.copy(),
                                 stdout=sink, stderr=subprocess.STDOUT,
                                 stdin=subprocess.DEVNULL, start_new_session=True)
        try:
            with logfile.open("r", encoding="utf-8", errors="replace") as reader:
                while True:
                    while True:
                        line = reader.readline()
                        if not line: break
                        if "E5_PROGRESS " in line or "INFERENCE_PROGRESS " in line:
                            last_progress = time.monotonic()
                        if line.strip(): print(f"[{name}] {line.rstrip()[:1800]}", flush=True)
                    now = time.monotonic()
                    if now-last_heartbeat >= 60:
                        gpu = subprocess.run(["nvidia-smi","--query-gpu=index,utilization.gpu,memory.used,memory.total",
                                              "--format=csv,noheader"], capture_output=True, text=True, timeout=15)
                        print("HEARTBEAT",name,"elapsed",round(now-start),"seconds; last progress",
                              round(now-last_progress),"seconds ago; GPU",gpu.stdout.strip(),flush=True)
                        last_heartbeat = now
                    if child.poll() is not None: break
                    if now-start > total_timeout or (inactivity and now-last_progress > inactivity):
                        raise TimeoutError(f"{name}: progress timeout; diagnostic traceback in {logfile}")
                    time.sleep(2)
        except BaseException:
            if child.poll() is None:
                os.killpg(child.pid, signal.SIGTERM)
                try: child.wait(timeout=20)
                except subprocess.TimeoutExpired:
                    os.killpg(child.pid, signal.SIGKILL)
                    child.wait()
            raise
    if child.returncode:
        raise RuntimeError(f"{name} failed ({child.returncode}); see {logfile}")
    print("DONE",name,"seconds",round(time.monotonic()-start),flush=True)


In [ ]:
subprocess.run([sys.executable,"-u","-m","pytest","-q","student_tests/test_student_e5.py"],cwd=ROOT,check=True)
for steps, limit in ((1,4),(5,20)):
    output = Path(f"/kaggle/working/e5_smoke_observable_{steps}")
    run_job(f"smoke_{steps}",["--mode","train","--base-exp","e3","--smoke",
            "--max-steps",str(steps),"--train-limit",str(limit),
            "--checkpoint-root",str(CHECKPOINT_ROOT),"--output-root",str(output),"--no-resume"],
            total_timeout=1800)
    run = json.loads((output/"run.json").read_text())
    assert run["steps"] == steps and run["max_completion_length"] == 32, run
    import math
    assert run["log_history"] and all(math.isfinite(float(row["loss"]))
           for row in run["log_history"] if "loss" in row), "Non-finite smoke loss"
print("SMOKE GATES PASSED",flush=True)


In [ ]:
run_job("full_60",["--mode","train","--base-exp","e3","--max-steps","60",
        "--checkpoint-root",str(CHECKPOINT_ROOT)], total_timeout=7200)
run = json.loads((ROOT/"report/evidence/e5_train_log.json").read_text())
assert run["steps"] == 60 and not run["smoke"] and run["max_completion_length"] == 128, run
assert run["train_images"] == 1422, run
print("FULL 60 TRAINING STEPS SAVED",flush=True)


In [ ]:
run_job("test_inference",["--mode","infer","--split","test","--checkpoint-root",str(CHECKPOINT_ROOT)],total_timeout=7200)
predictions = [json.loads(x) for x in (ROOT/"work/e5_test.jsonl").read_text().splitlines() if x.strip()]
assert len(predictions) == len({x["image_id"] for x in predictions}) == 311
run_job("analysis",["--mode","analyze","--baseline-exp","e3"],inactivity=None,total_timeout=7200)
for name in ("e5_train_log.json","e5_completion_samples.jsonl","e5.json","e5_analysis.json"):
    p = ROOT/"report/evidence"/name
    assert p.is_file() and p.stat().st_size > 0, p
    print("EVIDENCE",name,p.stat().st_size,"bytes",flush=True)
shutil.make_archive("/kaggle/working/e5_bonus_evidence","zip",ROOT/"report/evidence")
print("E5 COMPLETE: /kaggle/working/e5_bonus_evidence.zip",flush=True)
